# Zara 옷 사진 크롤러 (Playwright)

목표: Zara 카테고리 페이지에서 **옷 사진 + 메타데이터**(상품명·가격)를 수집한다.

진행 단계:
1. **정찰(Recon)** — 브라우저로 Zara 페이지를 열어 실제 구조(셀렉터)를 확인
2. **추출(Extract)** — 상품 카드에서 이름·가격·이미지 URL 뽑기
3. **수집(Collect)** — 무한스크롤로 100~200개까지 모으기
4. **저장(Save)** — 이미지 다운로드 + SQLite에 메타데이터 저장

> ⚠️ 학습/연구용. 요청 간 딜레이를 두고 소량만 받는다. 받은 이미지는 재배포하지 않는다.

In [26]:
# 셀 2: import + Windows 이벤트 루프 처리
import sys
import asyncio

# ── Windows 함정 해결 ──────────────────────────────────────────────
# Jupyter는 Windows에서 SelectorEventLoop를 쓰는데, Playwright는 subprocess를
# 띄우려고 ProactorEventLoop가 필요하다 → 안 맞으면 NotImplementedError.
# 아래 한 줄로 Proactor 정책을 강제한다. (반드시 playwright import 전에)
if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
# ───────────────────────────────────────────────────────────────────

from playwright.async_api import async_playwright

# 표준 라이브러리만 사용 (추가 설치 불필요)
import sqlite3
import json
from pathlib import Path
from datetime import datetime

print("import OK — event loop policy:", type(asyncio.get_event_loop_policy()).__name__)

import OK — event loop policy: WindowsProactorEventLoopPolicy


In [27]:
# 셀 2.5: Playwright 실행 헬퍼 (Windows + Jupyter 충돌 우회)
# 핵심: Jupyter의 기존 이벤트 루프는 건드리지 않고,
#       별도 "스레드" 안에서 새 ProactorEventLoop를 만들어 거기서 코루틴을 돌린다.
import threading

def run_async(coro_factory):
    """coro_factory: 인자 없이 코루틴을 새로 만들어 반환하는 함수 (예: lambda: recon())
    별도 스레드에서 전용 Proactor 루프로 실행하고 결과를 돌려준다."""
    result = {}
    def worker():
        loop = asyncio.new_event_loop()          # 이 스레드 전용 새 루프
        asyncio.set_event_loop(loop)
        try:
            result["value"] = loop.run_until_complete(coro_factory())
        except Exception as e:
            result["error"] = e
        finally:
            loop.close()
    t = threading.Thread(target=worker)
    t.start()
    t.join()                                     # 끝날 때까지 대기 (동기처럼 사용)
    if "error" in result:
        raise result["error"]
    return result.get("value")

print("run_async 준비됨 — 앞으로 Playwright 코루틴은 run_async(lambda: 함수()) 로 실행")

run_async 준비됨 — 앞으로 Playwright 코루틴은 run_async(lambda: 함수()) 로 실행


In [13]:
# 셀 3: 정찰 — 브라우저를 "눈에 보이게" 띄워서 Zara 구조 확인
# 크롤러가 아니라 "탐색용". 어느 셀렉터가 상품 카드를 잡는지 개수로 확인한다.

CATEGORY_URL = "https://www.zara.com/kr/ko/woman-dresses-l1066.html"

async def recon():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()

        await page.goto(CATEGORY_URL, wait_until="domcontentloaded")
        await page.wait_for_timeout(6000)  # JS 렌더링 + 쿠키창 뜰 시간

        print("페이지 제목:", await page.title())
        print("현재 URL :", page.url)  # 리다이렉트됐는지 확인

        # 상품 카드 후보 셀렉터들 — 0이 아니라 24개쯤 잡히는 게 진짜다.
        candidates = [
            "li.product-grid-product",
            "li[class*=product]",
            "article",
            "a[href*='/p/']",
            "ul.product-grid li",
            "img",  # 페이지에 이미지가 아예 뜨는지 sanity check
        ]
        print("\n--- 셀렉터별 매칭 개수 ---")
        for sel in candidates:
            try:
                n = await page.locator(sel).count()
            except Exception as e:
                n = f"err({e})"
            print(f"  {sel:32} → {n}")

        # 눈으로 볼 시간 (input 대신 고정 대기 — 스레드에서 input은 멈춤)
        print("\n>>> 8초간 브라우저를 열어둠. 쿠키창/상품을 눈으로 확인하세요.")
        await page.wait_for_timeout(8000)
        await browser.close()

run_async(lambda: recon())

페이지 제목: 여성 원피스 | ZARA 대한민국
현재 URL : https://www.zara.com/kr/ko/woman-dresses-l1066.html

--- 셀렉터별 매칭 개수 ---
  li.product-grid-product          → 31
  li[class*=product]               → 188
  article                          → 0
  a[href*='/p/']                   → 0
  ul.product-grid li               → 0
  img                              → 34

>>> 8초간 브라우저를 열어둠. 쿠키창/상품을 눈으로 확인하세요.


In [14]:
# 셀 4: 정찰 2단계 — 상품 카드 1개의 내부 구조 뜯어보기
# 확정된 카드 셀렉터: li.product-grid-product
# 이 카드 "안에서" 이름/가격/이미지/링크가 각각 어디 있는지 후보 셀렉터로 찾는다.

CARD = "li.product-grid-product"

async def inspect_card():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        await page.goto(CATEGORY_URL, wait_until="domcontentloaded")
        await page.wait_for_timeout(6000)

        first = page.locator(CARD).first  # 첫 번째 카드만

        # 카드 안 후보 셀렉터별로 "처음 발견된 텍스트/속성"을 출력해 본다.
        probes = {
            "이름 후보": [
                "a.product-link", "h2", "h3",
                "[class*=name]", "a[class*=link]",
            ],
            "가격 후보": [
                "[class*=price]", "span.price-current__amount", "span.money-amount__main",
            ],
            "링크 후보(href)": [
                "a.product-link", "a[href*='.html']", "a",
            ],
            "이미지 후보(src)": [
                "img.media-image__image", "img[src]", "picture img", "img",
            ],
        }

        for label, sels in probes.items():
            print(f"\n## {label}")
            for sel in sels:
                loc = first.locator(sel).first
                try:
                    n = await first.locator(sel).count()
                    if n == 0:
                        print(f"  {sel:34} → 0개")
                        continue
                    if "이미지" in label or "링크" in label:
                        attr = "src" if "이미지" in label else "href"
                        val = await loc.get_attribute(attr)
                    else:
                        val = (await loc.inner_text())[:50]
                    print(f"  {sel:34} → ({n}개) {val!r}")
                except Exception as e:
                    print(f"  {sel:34} → err: {e}")

        await page.wait_for_timeout(3000)
        await browser.close()

run_async(lambda: inspect_card())


## 이름 후보
  a.product-link                     → (1개) ''
  h2                                 → 0개
  h3                                 → 0개
  [class*=name]                      → 0개
  a[class*=link]                     → (1개) ''

## 가격 후보
  [class*=price]                     → 0개
  span.price-current__amount         → 0개
  span.money-amount__main            → 0개

## 링크 후보(href)
  a.product-link                     → (1개) 'https://www.zara.com/kr/ko/%E1%84%92%E1%85%A5%E1%84%82%E1%85%B5%E1%84%8F%E1%85%A9%E1%86%B7-%E1%84%86%E1%85%B5%E1%84%83%E1%85%B5-%E1%84%8B%E1%85%AF%E1%86%AB%E1%84%91%E1%85%B5%E1%84%89%E1%85%B3-p01003055.html'
  a[href*='.html']                   → (1개) 'https://www.zara.com/kr/ko/%E1%84%92%E1%85%A5%E1%84%82%E1%85%B5%E1%84%8F%E1%85%A9%E1%86%B7-%E1%84%86%E1%85%B5%E1%84%83%E1%85%B5-%E1%84%8B%E1%85%AF%E1%86%AB%E1%84%91%E1%85%B5%E1%84%89%E1%85%B3-p01003055.html'
  a                                  → (1개) 'https://www.zara.com/kr/ko/%E1%84%92%E1%85%A5%E1%84%82%E1%85%B5%E1%

In [15]:
# 셀 5: 정찰 3단계 — 첫 카드의 raw HTML을 직접 본다 (이름/가격 위치 찾기)
# 추측을 멈추고, 카드 안의 실제 HTML·모든 텍스트·이미지 alt를 출력한다.

async def dump_card_html():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        await page.goto(CATEGORY_URL, wait_until="domcontentloaded")
        await page.wait_for_timeout(6000)

        first = page.locator(CARD).first

        # (1) 카드 안 모든 텍스트 — 이름/가격이 텍스트로 있으면 여기 보인다
        full_text = await first.inner_text()
        print("=== 카드 전체 텍스트 ===")
        print(repr(full_text))

        # (2) 이미지 alt — Zara는 상품명을 alt에 넣는 경우가 많다
        alt = await first.locator("img").first.get_attribute("alt")
        print("\n=== 이미지 alt ===")
        print(repr(alt))

        # (3) raw HTML (앞 1800자만) — 클래스명을 눈으로 확인
        html = await first.inner_html()
        print("\n=== 카드 inner HTML (앞부분) ===")
        print(html[:1800])

        await page.wait_for_timeout(2000)
        await browser.close()

run_async(lambda: dump_card_html())

=== 카드 전체 텍스트 ===
''

=== 이미지 alt ===
'레드 롱 드레스 세트, 스트레이트 네크라인에 앞면 리본 디테일, 셔링 처리된 바디와 플레어 스커트.'

=== 카드 inner HTML (앞부분) ===
<div class="product-grid-product__figure"><a class="product-link product-grid-product__link link" data-qa-action="product-click" draggable="false" href="https://www.zara.com/kr/ko/%E1%84%92%E1%85%A5%E1%84%82%E1%85%B5%E1%84%8F%E1%85%A9%E1%86%B7-%E1%84%86%E1%85%B5%E1%84%83%E1%85%B5-%E1%84%8B%E1%85%AF%E1%86%AB%E1%84%91%E1%85%B5%E1%84%89%E1%85%B3-p01003055.html" tabindex="0"><div class="products-category-grid-media-container"><div class="media products-category-grid-media"><div class="media__wrapper media__wrapper--fill" style="padding-bottom:150%"><img alt="레드 롱 드레스 세트, 스트레이트 네크라인에 앞면 리본 디테일, 셔링 처리된 바디와 플레어 스커트." class="media-image__image media__wrapper--media" data-qa-qualifier="media-image" src="https://static.zara.net/assets/public/3aaa/19f1/676a4d99bccf/3fd35d2426ac/01003055964-p/01003055964-p.jpg?ts=1781707975291&amp;w=549"></div></div></div></a></div><div clas

In [24]:
# 셀 6: 추출 함수 — 한 페이지의 모든 카드에서 메타데이터를 뽑는다 (다운로드/스크롤은 아직 X)
import re

# 정찰로 확정된 셀렉터
CARD_SEL = "li.product-grid-product"
IMG_SEL  = "img.media-image__image"
LINK_SEL = "a.product-link"

def product_id_from_url(url: str) -> str | None:
    """상품 링크 URL 끝의 'p01003055.html' 에서 '01003055' 를 뽑는다."""
    m = re.search(r"-p(\d+)\.html", url or "")
    return m.group(1) if m else None

def is_real_image(url: str | None) -> bool:
    """진짜 상품 사진인지 판별. Zara는 lazy-load 전엔 투명 png placeholder를 넣는다."""
    if not url:
        return False
    if url.startswith("data:"):
        return False
    if "transparent-background" in url:   # ← lazy-load placeholder
        return False
    # Zara 상품컷은 static.zara.net 의 .jpg
    return "static.zara.net/assets" in url and ".jpg" in url

async def extract_products(page, category_label: str) -> list[dict]:
    """현재 페이지에 렌더된 카드 중 '진짜 이미지가 로드된' 것만 추출."""
    cards = page.locator(CARD_SEL)
    n = await cards.count()
    items = []
    for i in range(n):
        card = cards.nth(i)
        img = card.locator(IMG_SEL).first
        if await img.count() == 0:
            continue

        image_url = await img.get_attribute("src")
        if not is_real_image(image_url):      # placeholder/빈 값 스킵
            continue

        name      = await img.get_attribute("alt")
        link_loc  = card.locator(LINK_SEL).first
        product_url = await link_loc.get_attribute("href") if await link_loc.count() else None

        items.append({
            "product_id": product_id_from_url(product_url),
            "name": (name or "").strip(),
            "image_url": image_url,
            "product_url": product_url,
            "category": category_label,
        })
    return items


# --- 테스트: 첫 화면에서 '진짜 이미지'가 몇 개나 잡히는지 (스크롤 전이라 적게 나오는 게 정상) ---
async def test_extract():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        await page.goto(CATEGORY_URL, wait_until="domcontentloaded")
        await page.wait_for_timeout(6000)

        items = await extract_products(page, "woman_dress")
        print(f"진짜 이미지 추출 수: {len(items)} (스크롤 전이라 적은 게 정상)\n")
        for it in items[:3]:
            print(json.dumps(it, ensure_ascii=False, indent=2))

        await page.wait_for_timeout(1500)
        await browser.close()
        return items

sample_items = run_async(lambda: test_extract())

진짜 이미지 추출 수: 1 (스크롤 전이라 적은 게 정상)

{
  "product_id": "01003055",
  "name": "레드 롱 드레스 세트, 스트레이트 네크라인에 앞면 리본 디테일, 셔링 처리된 바디와 플레어 스커트.",
  "image_url": "https://static.zara.net/assets/public/3aaa/19f1/676a4d99bccf/3fd35d2426ac/01003055964-p/01003055964-p.jpg?ts=1781707975291&w=549",
  "product_url": "https://www.zara.com/kr/ko/%E1%84%92%E1%85%A5%E1%84%82%E1%85%B5%E1%84%8F%E1%85%A9%E1%86%B7-%E1%84%86%E1%85%B5%E1%84%83%E1%85%B5-%E1%84%8B%E1%85%AF%E1%86%AB%E1%84%91%E1%85%B5%E1%84%89%E1%85%B3-p01003055.html",
  "category": "woman_dress"
}


In [25]:
# 셀 7: 무한스크롤 수집 — 목표 개수까지 스크롤하며 진짜 이미지를 모은다
# 핵심 아이디어:
#   1) 조금 스크롤 → lazy-load가 placeholder를 진짜 .jpg로 바꿈
#   2) 현재 화면의 진짜 이미지들을 추출해서 dict에 누적 (product_id로 중복 제거)
#   3) 목표 개수 도달 또는 "더 스크롤해도 안 늘어남"이면 종료

async def scroll_and_collect(page, category_label: str, target: int = 120,
                             max_rounds: int = 60, pause_ms: int = 1200) -> list[dict]:
    collected: dict[str, dict] = {}   # product_id → item (중복 자동 제거)
    stagnant = 0                      # 새 항목이 안 늘어난 연속 횟수

    for round_i in range(max_rounds):
        # (1) 현재 화면에서 진짜 이미지 추출
        items = await extract_products(page, category_label)
        before = len(collected)
        for it in items:
            key = it["product_id"] or it["image_url"]   # id 없으면 url로 키
            collected[key] = it
        gained = len(collected) - before

        print(f"  round {round_i+1:2d}: 화면 {len(items):3d}개 → 누적 {len(collected):3d}개 (+{gained})")

        if len(collected) >= target:
            print(f"  ✅ 목표 {target}개 도달")
            break

        # (2) 새 항목이 없으면 stagnant 증가, 3번 연속이면 끝(페이지 바닥)
        stagnant = stagnant + 1 if gained == 0 else 0
        if stagnant >= 3:
            print("  ⛔ 더 스크롤해도 안 늘어남 → 종료 (카테고리 끝)")
            break

        # (3) 한 화면 높이만큼 스크롤 → 잠깐 대기 (lazy-load + 서버 배려)
        await page.mouse.wheel(0, 2500)
        await page.wait_for_timeout(pause_ms)

    return list(collected.values())


# --- 실행: 무한스크롤로 실제 수집 (target은 일단 60으로 테스트) ---
async def run_collect(category_url: str, category_label: str, target: int):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        await page.goto(category_url, wait_until="domcontentloaded")
        await page.wait_for_timeout(6000)

        items = await scroll_and_collect(page, category_label, target=target)
        print(f"\n총 수집: {len(items)}개")

        await page.wait_for_timeout(1500)
        await browser.close()
        return items

# 먼저 60개로 작게 테스트 (잘 되면 셀 8에서 120으로)
collected_items = run_async(lambda: run_collect(CATEGORY_URL, "woman_dress", target=60))

  round  1: 화면   1개 → 누적   1개 (+1)
  round  2: 화면   5개 → 누적   5개 (+4)
  round  3: 화면   7개 → 누적   7개 (+2)
  round  4: 화면   7개 → 누적   7개 (+0)
  round  5: 화면  10개 → 누적  10개 (+3)
  round  6: 화면  13개 → 누적  13개 (+3)
  round  7: 화면  15개 → 누적  15개 (+2)
  round  8: 화면  17개 → 누적  17개 (+2)
  round  9: 화면  19개 → 누적  19개 (+2)
  round 10: 화면  20개 → 누적  20개 (+1)
  round 11: 화면  22개 → 누적  22개 (+2)
  round 12: 화면  23개 → 누적  23개 (+1)
  round 13: 화면  27개 → 누적  27개 (+4)
  round 14: 화면  31개 → 누적  31개 (+4)
  round 15: 화면  35개 → 누적  34개 (+3)
  round 16: 화면  37개 → 누적  36개 (+2)
  round 17: 화면  38개 → 누적  37개 (+1)
  round 18: 화면  40개 → 누적  39개 (+2)
  round 19: 화면  41개 → 누적  40개 (+1)
  round 20: 화면  42개 → 누적  41개 (+1)
  round 21: 화면  44개 → 누적  41개 (+0)
  round 22: 화면  45개 → 누적  42개 (+1)
  round 23: 화면  48개 → 누적  43개 (+1)
  round 24: 화면  50개 → 누적  45개 (+2)
  round 25: 화면  53개 → 누적  48개 (+3)
  round 26: 화면  57개 → 누적  52개 (+4)
  round 27: 화면  58개 → 누적  53개 (+1)
  round 28: 화면  62개 → 누적  56개 (+3)
  round 29: 화면  66개 

In [28]:
# 셀 8: 저장 폴더 + SQLite 테이블 준비
# 폴더 구조: data/images/<category>/<product_id>.jpg  (폴더명 = 분류 라벨)
# 메타데이터: data/zara.db 의 products 테이블

DATA_DIR  = Path("data")
IMAGES_DIR = DATA_DIR / "images"
DB_PATH   = DATA_DIR / "zara.db"

IMAGES_DIR.mkdir(parents=True, exist_ok=True)

def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS products (
            product_id   TEXT PRIMARY KEY,   -- 중복 저장 방지 (같은 상품 다시 받아도 1행)
            name         TEXT,
            category     TEXT,
            image_url    TEXT,
            product_url  TEXT,
            image_path   TEXT,               -- 로컬에 저장된 파일 경로
            scraped_at   TEXT
        )
    """)
    conn.commit()
    conn.close()

init_db()
print(f"준비 완료\n  이미지 폴더: {IMAGES_DIR.resolve()}\n  DB: {DB_PATH.resolve()}")

준비 완료
  이미지 폴더: C:\dev\sesac\sesac-data-crawling\data\images
  DB: C:\dev\sesac\sesac-data-crawling\data\zara.db


In [29]:
# 셀 9: 이미지 다운로드(httpx) + 메타데이터 SQLite 저장
import httpx
import time

def save_metadata(item: dict, image_path: str):
    conn = sqlite3.connect(DB_PATH)
    conn.execute("""
        INSERT OR REPLACE INTO products
            (product_id, name, category, image_url, product_url, image_path, scraped_at)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        item["product_id"], item["name"], item["category"],
        item["image_url"], item["product_url"], image_path,
        datetime.now().isoformat(timespec="seconds"),
    ))
    conn.commit()
    conn.close()

def download_items(items: list[dict], delay: float = 0.5):
    """이미지를 내려받아 파일로 저장하고, 메타데이터를 DB에 기록."""
    headers = {  # 일반 브라우저처럼 보이게
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0 Safari/537.36",
        "Referer": "https://www.zara.com/",
    }
    ok, skipped, failed = 0, 0, 0
    with httpx.Client(headers=headers, timeout=20, follow_redirects=True) as client:
        for it in items:
            pid = it["product_id"]
            if not pid:
                skipped += 1
                continue

            cat_dir = IMAGES_DIR / it["category"]
            cat_dir.mkdir(parents=True, exist_ok=True)
            path = cat_dir / f"{pid}.jpg"

            if path.exists():          # 이미 받은 건 건너뜀 (재실행 안전)
                save_metadata(it, str(path))
                skipped += 1
                continue

            try:
                r = client.get(it["image_url"])
                r.raise_for_status()
                path.write_bytes(r.content)
                save_metadata(it, str(path))
                ok += 1
                print(f"  ✅ {pid}.jpg  ({len(r.content)//1024} KB)  {it['name'][:25]}")
            except Exception as e:
                failed += 1
                print(f"  ❌ {pid}: {e}")

            time.sleep(delay)          # 서버 배려 (요청 간 딜레이)

    print(f"\n완료 — 신규 {ok} / 스킵 {skipped} / 실패 {failed}")

# 셀 7에서 모은 collected_items 를 저장
download_items(collected_items, delay=0.5)

  ✅ 01003055.jpg  (58 KB)  레드 롱 드레스 세트, 스트레이트 네크라인에 
  ✅ 06929189.jpg  (24 KB)  가는 스트랩과 보트넥이 있는 짧은 라이트 블루
  ✅ 00881324.jpg  (26 KB)  짧은 라일락색 스트랩 드레스 세트, 밑단 프릴
  ✅ 04437055.jpg  (31 KB)  화이트 도트 패턴의 블랙 미디 드레스 세트, 
  ✅ 05644026.jpg  (11 KB)  라운드넥 긴팔 블랙 롱 원피스 세트
  ✅ 02359322.jpg  (12 KB)  비대칭 밑단, 라운드넥, 백 브이넥의 비대칭 
  ✅ 08741060.jpg  (14 KB)  가는 스트랩과 플레어 스커트가 있는 긴 블랙 
  ✅ 06929484.jpg  (26 KB)  카라, 짧은 소매, 플랩 프론트 포켓이 있는 
  ✅ 05039406.jpg  (48 KB)  레오파드 프린트, 스파게티 스트랩, V넥, 앞
  ✅ 05029261.jpg  (30 KB)  홀터넥, 브이넥, 셔링 디테일 허리선과 앞면 
  ✅ 00251784.jpg  (18 KB)  긴팔과 앞주머니가 있는 파란색 체크 미디 드레
  ✅ 02515909.jpg  (13 KB)  앞 버튼, 퍼프 롱 슬리브, 가슴 주름, 플랫
  ✅ 05029117.jpg  (13 KB)  옆트임이 있는 갈색 반팔 원피스 세트.
  ✅ 02201957.jpg  (15 KB)  아이보리 미디 드레스 세트, 브이넥, 짧은 퍼
  ✅ 04786065.jpg  (12 KB)  칼라, 옆트임, 사이드 포켓이 있는 화이트 반
  ✅ 05289025.jpg  (11 KB)  라운드 넥, 짧은 퍼프 소매, 옆면 포켓 디테
  ✅ 04772328.jpg  (34 KB)  민소매 스톤 드레스 세트, 라운드 넥, 숨겨진
  ✅ 02176669.jpg  (27 KB)  카키색 드레이프 백리스 탑, 와이드 팬츠, 플
  ✅ 03897118.jpg  (50 KB)  검은색 물방울 무늬, 오프숄더 네크라인, 얇은
  ✅